In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

In [3]:
#Load data
df = pd.read_csv("/content/feature_engineered_data.csv")

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce",
    format="mixed"
)

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Year,Month,MonthName,Day,DayOfWeek,Hour,Quarter,IsWeekend,BasketSize,OrderValue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009,12,December,1,Tuesday,7,4,0,166,505.3
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,December,1,Tuesday,7,4,0,166,505.3
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,December,1,Tuesday,7,4,0,166,505.3
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009,12,December,1,Tuesday,7,4,0,166,505.3
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009,12,December,1,Tuesday,7,4,0,166,505.3


In [14]:
#Creat churn label
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

customer = df.groupby("Customer ID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "Invoice":"nunique",
    "Revenue":"sum"
}).reset_index()

customer.columns = [
    "CustomerID",
    "Recency",
    "Frequency",
    "Monetary"
]

# Business Logic
customer["Churn"] = np.where(customer["Recency"] >= 15, 1, 0)


customer.head()

,CustomerID,Recency,Frequency,Monetary,Churn
0,12346.0,2,5,113.50,0
1,12358.0,13,1,1429.83,0
2,12359.0,4,2,838.89,0
3,12362.0,19,1,130.00,1
4,12417.0,10,2,317.60,0


In [15]:
customer["Churn"].value_counts()

,count
Churn,
0,701
1,233


In [16]:
#Feature Target
X = customer[["Recency", "Frequency", "Monetary"]]
y = customer["Churn"]

In [17]:
#Train/ Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
#random forcast model
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [19]:
#prediction
y_pred = model.predict(X_test)

In [20]:
#Acccuracy
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy :", accuracy_score(y_test, y_pred))
print()

print(classification_report(y_test, y_pred))

Accuracy : 1.0

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       140
           1       1.00      1.00      1.00        47

    accuracy                           1.00       187
   macro avg       1.00      1.00      1.00       187
weighted avg       1.00      1.00      1.00       187



In [21]:
import pickle

with open("churn_model.pkl", "wb") as f:
    pickle.dump(model, f)

customer.to_csv("customer_churn_prediction.csv", index=False)

print("✅ Churn Prediction Completed")

✅ Churn Prediction Completed
